# Demo Modul 2: Backpropagation dan Automatic Differentiation

**Durasi sesi:** 120 menit  
**Kasus:** jaringan $2 \rightarrow 2 \rightarrow 1$ yang sama persis dengan Modul 1, dihitung mundur sampai gradien.

Notebook ini dijalankan bersama asisten. Notebook tugas ada pada `starter-mahasiswa.ipynb`.

## Capaian demo

Setelah demo, praktikan dapat:

1. membaca graf komputasi dan menandai gradien lokal setiap simpul;
2. menulis backward manual yang mengembalikan gradien keempat parameter;
3. mencocokkan hasilnya dengan autograd dan menjelaskan `zero_grad()`;
4. memverifikasi gradien dengan finite difference; dan
5. mengenali kesalahan training loop yang tidak memunculkan pesan galat.

In [1]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)   # pemeriksaan gradien butuh float64
pd.set_option('display.precision', 8)
print({
    'python': platform.python_version(),
    'numpy': np.__version__,
    'torch': torch.__version__,
    'device': str(DEVICE),
})

{'python': '3.13.15', 'numpy': '2.1.3', 'torch': '2.11.0+cpu', 'device': 'cpu'}


## 1. Kasus 1: forward pass yang sudah dikenal

Nilai berikut tetap dan tidak diubah sepanjang modul.

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Hidden layer memakai ReLU, keluaran memakai BCE-with-logits.

In [2]:
x  = np.array([2.0, -1.0])
W1 = np.array([[0.5, -0.5], [1.0, 1.0]])
b1 = np.array([0.0, 0.0])
W2 = np.array([2.0, -1.0])
b2 = 0.5
y  = 1.0

def forward(x, W1, b1, W2, b2, y):
    """Mengembalikan seluruh nilai perantara, bukan hanya loss."""
    z1 = W1 @ x + b1
    h  = np.maximum(z1, 0.0)
    z2 = W2 @ h + b2
    p  = 1.0 / (1.0 + np.exp(-z2))
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

nilai = forward(x, W1, b1, W2, b2, y)
for nama, v in nilai.items():
    print(f'{nama:>5}: {np.round(v, 6)}')

   z1: [1.5 1. ]
    h: [1.5 1. ]
   z2: 2.5
    p: 0.924142
 loss: 0.07889


**Pemeriksaan:** hasilnya harus $\mathbf{z}^{(1)}=[1{,}5\;\;1]$, $\mathbf{h}=[1{,}5\;\;1]$, logit $2{,}5$, $p \approx 0{,}924142$, dan $\mathcal{L} \approx 0{,}078890$.

Karena kedua pra-aktivasi bernilai positif, ReLU meneruskan keduanya apa adanya. Ingat fakta ini: nanti ia menentukan gradien mana yang lolos dan mana yang diblokir.

## 2. Backward manual

Urutan penurunannya mengikuti graf dari belakang:

$$\frac{\partial \mathcal{L}}{\partial z^{(2)}} = p - y
\;\longrightarrow\;
\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{(2)}},\;
\frac{\partial \mathcal{L}}{\partial b^{(2)}},\;
\frac{\partial \mathcal{L}}{\partial \mathbf{h}}
\;\longrightarrow\;
\frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(1)}}
\;\longrightarrow\;
\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{(1)}},\;
\frac{\partial \mathcal{L}}{\partial \mathbf{b}^{(1)}}$$

In [3]:
def backward(x, W1, W2, y, nilai):
    z1, h, p = nilai['z1'], nilai['h'], nilai['p']

    dz2 = p - y                      # turunan BCE-with-logits terhadap logit
    dW2 = dz2 * h                    # gradien lokal linear: h
    db2 = dz2                        # gradien lokal linear: 1
    dh  = dz2 * W2                   # gradien lokal linear: W2

    dz1 = dh * (z1 > 0)              # gradien lokal ReLU: 1 bila z1 > 0
    dW1 = np.outer(dz1, x)           # baris ke-i dikalikan x
    db1 = dz1

    return {'dz2': dz2, 'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

grad_manual = backward(x, W1, W2, y, nilai)
for nama, v in grad_manual.items():
    print(f'{nama:>4}: {np.round(v, 7)}')

 dz2: -0.0758582
  W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
  b1: [-0.1517164  0.0758582]
  W2: [-0.1137873 -0.0758582]
  b2: -0.0758582


**Pemeriksaan:** `dz2` $\approx -0{,}0758582$ dan `W2` $\approx [-0{,}1137873\;\; -0{,}0758582]$.

Perhatikan shape: setiap gradien berbentuk sama dengan parameter pasangannya. `dW1` berukuran $2\times2$ seperti $\mathbf{W}^{(1)}$, bukan $2\times1$.

## 3. Autograd mengerjakan hal yang sama

Yang berubah hanya siapa yang menelusuri graf. Persamaannya identik.

In [4]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx  = torch.tensor(x)
ty  = torch.tensor(y)

kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    z1 = tW1 @ tx + tb1
    h  = torch.relu(z1)
    z2 = tW2 @ h + tb2
    return kriteria(z2, ty)          # loss menerima LOGIT, bukan probabilitas

loss = forward_torch()
loss.backward()

for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    print(f'{nama}: {np.round(t.grad.numpy(), 7)}')

W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
b1: [-0.1517164  0.0758582]
W2: [-0.1137873 -0.0758582]
b2: -0.0758582


## 4. Mengapa `zero_grad()` diperlukan

`backward()` **menambahkan** ke `.grad`, bukan menimpanya. Perhatikan apa yang terjadi bila dipanggil dua kali.

In [5]:
forward_torch().backward()        # panggilan kedua, tanpa menghapus gradien
print('setelah backward kedua :', np.round(tW2.grad.numpy(), 7))

for t in (tW1, tb1, tW2, tb2):    # cara menghapusnya secara manual
    t.grad = None

forward_torch().backward()
print('setelah gradien dihapus:', np.round(tW2.grad.numpy(), 7))

setelah backward kedua : [-0.2275745 -0.1517164]
setelah gradien dihapus: [-0.1137873 -0.0758582]


Gradien pada panggilan kedua tepat dua kali lipat. Pada training loop, perilaku inilah yang menjadi kesalahan paling sering: tanpa `optimizer.zero_grad()`, langkah pembaruan memakai jumlah gradien seluruh batch sebelumnya.

## 5. Gradient checking

Gradien analitik dibandingkan dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

In [6]:
EPS = 1e-5

def loss_dengan(param, i, delta):
    """Menghitung loss setelah menggeser satu komponen parameter sebesar delta."""
    p = {'W1': W1.copy(), 'b1': b1.copy(), 'W2': W2.copy(), 'b2': np.array(b2)}
    p[param][i] += delta
    return forward(x, p['W1'], p['b1'], p['W2'], float(p['b2']), y)['loss']

def finite_difference(param, i):
    return (loss_dengan(param, i, EPS) - loss_dengan(param, i, -EPS)) / (2 * EPS)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.numpy()
    auto = auto[i] if i != () else auto
    numerik = finite_difference(nama, i if i != () else ())
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))
print('\nrelative error maksimum:', tabel['rel_err'].max())

parameter indeks      manual    autograd     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 -0.30343272 1.00453007e-10
       W1 (0, 1)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 0)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 1) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b1   (0,) -0.15171636 -0.15171636 -0.15171636 2.95622632e-11
       b1   (1,)  0.07585818  0.07585818  0.07585818 5.73360674e-11
       W2   (0,) -0.11378727 -0.11378727 -0.11378727 6.76755052e-11
       W2   (1,) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b2     () -0.07585818 -0.07585818 -0.07585818 5.73360674e-11

relative error maksimum: 1.004530066507852e-10


**Pemeriksaan:** seluruh `rel_err` harus di bawah $10^{-5}$.

Dua hal yang membuat pemeriksaan ini gagal padahal gradiennya benar: memakai `float32`, dan memilih $\epsilon$ terlalu kecil sehingga selisih dua bilangan besar kehilangan presisi.

## 6. Kesalahan yang tidak memunculkan pesan galat

Contoh berikut sengaja memakai satu kesalahan saja: `zero_grad()` dihilangkan. Kodenya berjalan, tidak ada peringatan, tetapi hasilnya salah.

In [7]:
def regresi_kecil(pakai_zero_grad: bool, epoch: int = 30):
    seed_everything(SEED)
    w = torch.tensor(0.0, requires_grad=True)
    data_x = torch.tensor([1.0, 2.0, 3.0, 4.0])
    data_y = 2.0 * data_x                      # jawaban benar: w = 2
    opt = torch.optim.SGD([w], lr=0.01)

    for _ in range(epoch):
        if pakai_zero_grad:
            opt.zero_grad()
        loss = ((w * data_x - data_y) ** 2).mean()
        loss.backward()
        opt.step()
    return w.item(), loss.item()

for pakai in (True, False):
    w_akhir, loss_akhir = regresi_kecil(pakai)
    label = 'dengan zero_grad ' if pakai else 'tanpa zero_grad  '
    print(f'{label}: w = {w_akhir:8.4f}   loss = {loss_akhir:10.4f}')

dengan zero_grad : w =   1.9847   loss =     0.0024
tanpa zero_grad  : w =   0.4131   loss =     7.2263


Tanpa `zero_grad()`, gradien menumpuk sehingga langkah pembaruan membesar tak terkendali dan nilai `w` melewati jawaban yang benar.

Pada notebook tugas, Anda akan menghadapi fungsi `train_rusak` untuk kasus XOR yang memuat **empat** kesalahan sejenis. Semuanya berjalan tanpa pesan galat.

## Exit ticket

1. Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$?
2. Pada Kasus 1, gradien mana yang akan menjadi nol bila $\mathbf{z}^{(1)}$ bernilai negatif?
3. Apa bedanya peran `backward()` dan `optimizer.step()`?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb`, lengkapi tabel relative error untuk seluruh parameter, lalu temukan dan perbaiki keempat kesalahan pada `train_rusak`.

### Jawaban Exit Ticket

#### 1. Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$?

Kalau kita runut rumusnya, loss function Binary Cross Entropy (BCE) itu kan:
$$\mathcal{L} = -(y \log(p) + (1 - y) \log(1 - p))$$

di mana nilai probabilitas $p$ didapat dari fungsi sigmoid terhadap logit $z^{(2)}$:
$$p = \sigma(z^{(2)}) = \frac{1}{1 + e^{-z^{(2)}}}$$

Nah, pas kita cari turunan $\mathcal{L}$ terhadap logit $z^{(2)}$ pakai aturan rantai (*chain rule*), kita pecah jadi dua bagian:
$$\frac{\partial \mathcal{L}}{\partial z^{(2)}} =
\underbrace{\frac{\partial \mathcal{L}}{\partial p}}_{\text{Bagian A}} \cdot \underbrace{\frac{\partial p}{\partial z^{(2)}}}_{\text{Bagian B}}$$

- **Bagian A** (turunan loss terhadap $p$):
$$\frac{\partial \mathcal{L}}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p} = \frac{p-y}{p(1-p)}$$

- **Bagian B** (turunan sigmoid terhadap logit):
$$\frac{\partial p}{\partial z^{(2)}} = p(1-p)$$

Pas kita kaliin keduanya, komponen $p(1-p)$ di pembilang dan penyebut bakal saling coret:
$$\frac{\partial \mathcal{L}}{\partial z^{(2)}} = \frac{p-y}{\cancel{p(1-p)}} \cdot \cancel{p(1-p)} = p - y$$

Hasil akhirnya jadi  ($p-y$). Bentuk ini secara komputasi juga jauh lebih aman dibanding kita hitung satu-satu, karena gak bakal kena masalah *division by zero* (pembagian dengan angka yang terlalu kecil).

---

#### 2. Pada Kasus 1, gradien mana yang akan menjadi nol bila $\mathbf{z}^{(1)}$ bernilai negatif?

Kalau pre-aktivasi $\mathbf{z}^{(1)}$ bernilai negatif (semua elemennya $< 0$), otomatis fungsi aktivasi **ReLU** bakal memotong nilainya jadi nol semua ($h_i = 0$). Karena ReLU mendatar di area negatif, turunan lokalnya juga otomatis jadi nol.

Di bagian backpropagation, kodenya ditulis:
```python
dz1 = dh * (z1 > 0)
```

Kalau semua elemen di `z1` negatif, kondisi `z1 > 0` bakal menghasilkan nilai False (atau 0). Akibatnya:
- `dz1` nilainya jadi nol semua `[0.0, 0.0]`.
- Imbasnya, gradien untuk parameter layer pertama, yaitu **`dW1`** (hasil perkalian luar `dz1` dengan input `x`) dan bias **`db1`** (yang nilainya sama dengan `dz1`), bakal **ikut jadi nol**.

Sederhananya, aliran gradien langsung mampet/mati di layer ReLU ini.

---

#### 3. Apa bedanya peran `backward()` dan `optimizer.step()`?



- **`backward()`** (misal `loss.backward()`): Tugasnya buat **menghitung gradien**. Dia bakal jalan mundur dari loss ke belakang lewat graf komputasi untuk mencari tahu seberapa besar pengaruh tiap parameter (`requires_grad=True`) terhadap error. Hasil hitungan gradien ini cuma disimpan (diakumulasi) di atribut `.grad` masing-masing variabel (misalnya di `W1.grad`). Di tahap ini, **belum ada perubahan nilai bobot sama sekali**.

- **`optimizer.step()`**: Tugasnya baru melakukan **update parameter/bobot**. Setelah gradiennya ketemu dan disimpan oleh `backward()`, optimizer bakal pakai nilai di `.grad` tersebut untuk mengubah bobot asli sesuai dengan algoritma yang dipakai (misal SGD atau Adam) dan ukuran *learning rate* yang sudah kita atur.